# Day 1.6 — Represent the Agent Loop with LangGraph

You have already written the loop, so nothing new happens today conceptually. LangGraph
gives the same mechanism an explicit shape — named nodes, typed state, and edges you can
draw:

```text
START -> model --final answer--> END
           |-- tool request --> tools --> model
           |-- step limit ----> limit --> END
```

LangGraph is not the agent's intelligence and it does not replace the provider call. It
organises execution, which starts to matter on Day 3 when we need branching, interrupts
and checkpoints.

## Before you begin

### Learning outcomes

- Name the four pieces of a graph: state, node, edge, conditional edge.
- Map each line of your Step 5 loop from 1.5 onto one of them.
- Run the graph and read the final state, including which route ended the run.

Architecture reference: [D05](../../diagrams/source/day_01.md).

### Expected observation

The compiled graph prints as a small diagram, the run ends with the same validated answer
as the manual loop, and `max_steps=1` exits through the `limit` node instead of `END`.

If `langgraph` is not installed, every cell prints an install hint and skips — nothing
raises, and notebooks 01-05 and 07 are unaffected.

## Concept briefing

## Workflow or agent?

Not every problem needs an agent. Use ordinary code or a deterministic workflow when the
steps and decision rules are known. Use a hybrid workflow when most steps are fixed but
one bounded judgment benefits from a model. Consider an agent when the next action cannot
be fully predetermined, the action set is small, failures are containable, and success
can be evaluated.

Ask:

1. Are the steps known in advance?
2. Can normal code make the decision reliably?
3. Does the model genuinely add judgment rather than decoration?
4. What is the consequence of a wrong action?
5. Is there a strict step and tool boundary?
6. Can we observe and evaluate the result?

If these questions have weak answers, the correct design is often a workflow, not an
agent.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/research_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

### Step 1 — Check that LangGraph is available

Optional dependencies are imported inside a `try`, so a missing package produces a printed
hint instead of a traceback. `GRAPH_AVAILABLE` then guards every later cell.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

try:
    # Imported only to prove the real API is present, not used directly in this cell.
    from langgraph.graph import StateGraph  # noqa: F401
    GRAPH_AVAILABLE = True
    try:
        installed = version("langgraph")
    except PackageNotFoundError:
        installed = "unknown"
    print("langgraph is installed - version", installed)
except ImportError:
    GRAPH_AVAILABLE = False
    print("Optional: run  pip install langgraph  (it is already in requirements.txt)")
    print("to run this notebook's graph cells. Every cell below will skip safely.")
    print("Nothing else in Day 1 needs it.")

### Step 2 — State is application-owned data

A graph node is an ordinary Python function: it receives the current state and returns the
fields it wants to change. Our state is the same information the manual loop kept in local
variables — the message list, a step counter, the limit, the validated answer, and an error.

In [ ]:
from research_agent.agent import SYSTEM_MESSAGE
from research_agent.providers import MockModelProvider, OpenRouterProvider
from research_agent.schemas import Message
from research_agent.tools import default_tool_registry

provider = OpenRouterProvider() if LIVE else MockModelProvider()
tools = default_tool_registry()
print("Provider:", type(provider).__name__)

question = "Explain an AI tool using the local notes and calculate 12 * 7."

initial_state = {
    "messages": [Message(role="system", content=SYSTEM_MESSAGE),
                 Message(role="user", content=question)],
    "steps": 0,               # incremented by the model node
    "max_steps": 5,           # read by the routing function
    "final_response": None,   # filled in when a validated answer arrives
    "error": None,            # filled in on validation failure or step limit
}

print()
print("Initial state:")
for key, value in initial_state.items():
    shown = [m.role for m in value] if key == "messages" else value
    print(f"  {key:15} = {shown}")

### Step 3 — Compile the graph and draw it

`build_graph` lives in `src/research_agent/graph.py`. It registers three nodes (`model`,
`tools`, `limit`), one conditional edge out of `model`, and a plain edge from `tools` back
to `model` — the arrow that makes it a loop.

In [ ]:
graph = None
if GRAPH_AVAILABLE:
    from research_agent.graph import build_graph
    graph = build_graph(provider, tools)
    print(graph.get_graph().draw_mermaid())
else:
    print("Skipped: langgraph is not installed.")

### Step 4 — Invoke it and read the final state

`graph.invoke` runs nodes until something routes to `END`. What comes back is the final
state dictionary, not just an answer — which is why a graph is easy to inspect.

In [ ]:
final_state = None
if graph is not None:
    final_state = graph.invoke(initial_state)
    print("steps taken :", final_state["steps"])
    print("error       :", final_state["error"])
    print()
    if final_state["final_response"]:
        print(final_state["final_response"].model_dump_json(indent=2))
    else:
        print("No validated final response.")
else:
    print("Skipped: langgraph is not installed.")

### Step 5 — The route, read from the messages

The message list tells you which path the run took. It is the same trace the manual loop
produced in 1.5, because it is the same loop.

In [ ]:
if final_state is not None:
    for index, message in enumerate(final_state["messages"]):
        requested = [call.name for call in message.tool_calls]
        preview = message.content[:60].replace("\n", " ")
        print(f"{index}. role={message.role:9} tool_requests={requested} content={preview!r}")
else:
    print("Skipped: langgraph is not installed.")

### Step 6 — Take the limit route

Setting `max_steps` to 1 sends the run through the `limit` node instead of `END`. The
graph does not crash; it records why it stopped, exactly as the manual loop did.

In [ ]:
if graph is not None:
    limited_state = graph.invoke({**initial_state, "max_steps": 1, "steps": 0})
    print("steps         :", limited_state["steps"])
    print("error         :", limited_state["error"])
    print("final_response:", limited_state["final_response"])
    print()
    print("Ended through the 'limit' node, so the error field explains the stop.")
else:
    print("Skipped: langgraph is not installed.")

### Step 7 — Manual loop and graph, side by side

| Manual loop (1.5, Step 5)          | Graph (this notebook)             |
|------------------------------------|-----------------------------------|
| `for step in range(...)`            | the `model` node, visited again   |
| `if not turn.tool_calls:`           | the conditional edge out of `model` |
| the tool execution block            | the `tools` node                  |
| `return {"status": "completed"}`    | the edge to `END`                 |
| `return {"status": "max_steps"}`    | the `limit` node, then `END`      |
| local variables                     | the state dictionary              |

Nothing about the architecture changed. What changed is that the control flow is now data
you can print, draw, pause and resume.

### Checkpoint

**1. If LangGraph is not installed, is anything about your Day 1 agent broken?**

<details><summary>Show answer</summary>

No. The agent is the loop in `src/research_agent/agent.py`, which imports nothing optional.
`graph.py` imports LangGraph inside its factory function precisely so notebooks 01-05 and
07 keep working without it. That is the general pattern for an optional dependency: import
it late, set a flag, and let the dependent cells print a hint and skip.

</details>

**2. What did LangGraph change, and what did it not change?**

<details><summary>Show answer</summary>

Changed: the control flow is explicit and inspectable — nodes, a typed state dictionary and
routing you can draw, pause and checkpoint. Not changed: the provider call, the tool
schemas, argument validation, the step limit, and the final-answer contract. Adding a graph
does not make a system more capable or safer; it makes its orchestration easier to see.

</details>

### Recap

- **Limitation we saw:** a hand-written loop is fine at this size, but its control flow
  only exists as local variables, so it cannot be drawn, paused or resumed.
- **Layer we added:** the same loop expressed as state, nodes and conditional edges — with
  a graceful skip when the optional package is missing.
- **Evidence it worked:** the graph printed its own diagram, produced the same validated
  answer as the manual loop, and `max_steps=1` exited through the `limit` node.